![NWM](img/NWM.png)

# Retrieve and Analyze National Water Model Snow Data for a Watershed of Interest
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org)
Last updated:Oct 16, 2025

### 1. Prepare the Python Environment

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import time

prefix = os.environ['CONDA_PREFIX']
os.environ['PROJ_LIB'] = os.path.join(prefix, 'share', 'proj')

import pyproj
import pandas as pd
import xarray as xr
import geopandas as gpd
from dask.distributed import Client

from utils import nwm_utils

We'll use dask to parallelize our code. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. 

In this setup, we create a Dask client with `Client(n_workers=6, threads_per_worker=1, memory_limit='2GB')`, which launches a cluster with 6 workers. Each worker uses a single thread, typically mapped to one CPU core, allowing for efficient parallel processing across 6 cores. Each worker also has a memory limit of 2 GB, for a total of up to 12 GB across the cluster.


In [ ]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print('Dashboard link:', client.dashboard_link)
except:    
    # The client should be customized to your workstation resources.
    client = Client(n_workers=6, threads_per_worker=1, memory_limit='2GB') 
    print('Dashboard link:', client.dashboard_link)
print(client)

### 2. Set Inputs

In [ ]:
# Path to the watershed shapefile
watershed = "./domain_data/TolumneRiver_18040009.shp"
basin = "./domain_data/DonPedroDam_Upstream_Basin.shp"

# Path to NWM snow data
conus_bucket_url = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/ldasout.zarr'

# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2020-09-30'

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './obs_outputs' 
MOD_OutputFolder = './mod_outputs'

### 3. Retrieve Observed Snow Data 

This code reads a GeoJSON file of all snow observation stations and filters them to include only stations in California with available CSV data. It also loads the watershed boundary shapefile (`TolumneRiver_18040009.shp`), converts it to the appropriate coordinate system, and merges all watershed polygons into a single MultiPolygon. Finally, it counts how many observation sites fall within this combined watershed boundary, providing the number of sites located inside the study area. 

**Heads up**: You might need to run the next cell twice. Sometimes, the spatial filtering does not return any results the first time due to how geometries are loaded or processed in the background.

In [ ]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]
filtered_all_stations_gdf = all_stations_gdf[all_stations_gdf.state.str.contains('California')]  

# Extract the bounding box coordinates of a watershed
watershed_gdf = gpd.read_file(os.path.join(os.getcwd(), watershed)).to_crs(epsg=4326)

# Combine all polygons into a single MultiPolygon
watershed_union = watershed_gdf.geometry.unary_union

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = filtered_all_stations_gdf[filtered_all_stations_gdf.geometry.within(watershed_union)]
gdf_in_bbox.reset_index(inplace=True)
print(f'There are {len(gdf_in_bbox)} sites from', ', '.join(set(gdf_in_bbox.network)), 'network(s) within the watershed.')


Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
## TODO: REPLACE WITH CSSI_EVALUATION.PLOTS FUNCTIONS

m = nwm_utils.plot_sites_within_domain(gdf_in_bbox, watershed_gdf, zoom_start=9)
m

Check the start and end date of available data for these sites.

In [ ]:
for i in gdf_in_bbox.index:
    print('Site ', gdf_in_bbox.iloc[i].code, ":", gdf_in_bbox.iloc[i].beginDate, "-", gdf_in_bbox.iloc[i].endDate)

The following uses the `nwm_utils.py` script to download observed data for the sites within the domain. Since all the sites are from the (California Cooperative Snow Survey) CCSS network, we use the `getCCSSData` function from the module to get data. 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Did you know?</h4>
<p>The California Cooperative Snow Surveys (CCSS) program, managed primarily by the California Department of Water Resources (DWR), monitors snowpack across key California watersheds to estimate annual water supply. Most CCSS sites are manual snow courses, where surveyors physically measure snow depth and snow water equivalent (SWE) several times a year, though some sites have been upgraded with automated sensors. In contrast, SNOTEL sites, managed by the USDA Natural Resources Conservation Service (NRCS), are fully automated and collect data continuously across the western United States, including California. While CCSS primarily supports water supply forecasting, SNOTEL supports both water supply forecasting and climate monitoring. </p>
</div>

In [ ]:
# Create a folder to save results
isExist = os.path.exists(OBS_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(OBS_OutputFolder)

In [ ]:
for i in gdf_in_bbox.index:
    nwm_utils.getCCSSData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], StartDate, EndDate, OBS_OutputFolder)

### 4. Retrieve Modeled Snow Data

NOAA shares inputs and outputs to the National Water Model retrospective simulations version 3 at <a href="https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html" style="color: blue; background-color: snow;">https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html</a>. The following code uses `fsspec` and `xarray` Python libraries to load the Zarr metadata of snow outputs (**ldasout.zarr**) into memory. Once the code is executed, you can see the wall time, which includes time spent waiting for I/O operations, such as reading data from a remote server. In our case, it took about 12 seconds to load the metadata into memory. Set up `Dask`, a parallel computing library, to enable performing operations on large datasets that don't fit into memory by breaking them into smaller, manageable pieces called chunks.

In [ ]:
%%time 
ds = xr.open_zarr(
    store=conus_bucket_url,
    consolidated=True,
    storage_options={
        "anon": True,
        "client_kwargs": {"region_name": "us-east-1"}
    }
)

The following code retrieves NWM data for each SNOTEL site within our watershed. For each site, it first converts latitude and longitude of the site to the projected coordinates used by the NWM. Then, it extracts the NWM data SWE for the site and the period of interest, saving the result as a DataFrame. Since NWM timestamps are in UTC, the DataFrame is converted to the local time zone to match SNOTEL observations for later comparison purposes. To fairly compare with SNOTEL, which reports SWE once daily at the start of the local day, the data is grouped by date, and the earliest record of each day is selected. Finally, the processed data is saved as a CSV file for each site.

In [ ]:
# Create a folder to save results
isExist = os.path.exists(MOD_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(MOD_OutputFolder)

In [ ]:
# Retrieve data for the location of snotel sites
input_crs = 'EPSG:4269'
output_crs = pyproj.CRS(ds.crs.esri_pe_string) 

for i in range(0, gdf_in_bbox.shape[0]):
    site_name = gdf_in_bbox.iloc[i]["name"]
    print(f'[{i+1}/{len(gdf_in_bbox)}] Retrieving data for site: {site_name}')
    snotel_y, snotel_x = nwm_utils.convert_latlon_to_yx(gdf_in_bbox.iloc[i].latitude, 
                                                      gdf_in_bbox.iloc[i].longitude, 
                                                        input_crs, ds, output_crs)
    
    dl_start_time = time.time()
    ds_subset = ds[['SNEQV']].sel(y=snotel_y, x=snotel_x, method='nearest'
                                 ).sel(time=slice(StartDate, EndDate)).compute()
    dl_elapsed = time.time() - dl_start_time
    print(f'✅ Retrieved data for {site_name} in {dl_elapsed:.2f} seconds\n')
    
    df = ds_subset.to_dataframe()
    df=df.drop(columns=['x', 'y'])
    df.reset_index(inplace=True)
    df["time"] = pd.to_datetime(df["time"])
    df.rename(columns={df.columns[0]:'Date', df.columns[1]:'NWM_SWE_meters'}, inplace=True)
    df.iloc[:, 1:] = df.iloc[:, 1:].apply(lambda x: pd.to_numeric(x)/1000)  # convert mm to m   

    # convert utc to local time zone
    df_local = nwm_utils.convert_utc_to_local(gdf_in_bbox.iloc[i].state, df)   
    
    # groupby the data and select the first item from each group 
    df_local.index = pd.to_datetime(df_local['Date_Local'])
    df_local = df_local.groupby(pd.Grouper(freq='D')).first()

    # save
    df_local.to_csv(f'./{MOD_OutputFolder}/df_{gdf_in_bbox.iloc[i].code}_{gdf_in_bbox.iloc[i].state}_NWM.csv', index=False)